# CIFAR-10 Image Classification -- From Scratch (Custom ResNet)

## S1 - Imports & Configuration

In [2]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT   = os.getenv('PROJECT_ROOT')
DATA_PATH      = os.getenv('DATA_PATH')
CHECKPOINT_DIR = os.getenv('CHECKPOINT_DIR', os.path.join(PROJECT_ROOT, 'checkpoints'))
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

import datetime
timestamp = datetime.datetime.now().strftime('%d%m-%H%M')

torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device:          {device}")
print(f"Project root:    {PROJECT_ROOT}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")

Device:          cuda
Project root:    Q:\scripts\projects\ComputerVision-Term9
Checkpoint dir:  Q:\scripts\projects\ComputerVision-Term9\checkpoints


## S2 - Data Loading & Transforms

In [3]:
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# CIFAR-10 native normalization stats
CIFAR10_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR10_STD  = [0.2470, 0.2435, 0.2616]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

# Full 50k training set with augmentation -- phase-specific loaders defined per section
full_train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_PATH, train=True, download=True, transform=train_transform
)

# Mirror dataset with val_test_transform for clean validation splits (no augmentation)
full_train_dataset_val = torchvision.datasets.CIFAR10(
    root=DATA_PATH, train=True, download=False, transform=val_test_transform
)

# Full 10k test set
test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_PATH, train=False, download=True, transform=val_test_transform
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False,
    num_workers=8, pin_memory=True, persistent_workers=True
)

print(f"Full training set: {len(full_train_dataset)}")
print(f"Test set:          {len(test_dataset)}")

Files already downloaded and verified


q:\scripts\projects\ComputerVision-Term9\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified
Full training set: 50000
Test set:          10000


## S3 - Model Architecture

In [4]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, planes, stride=1, downsample=None, dropout=0.1):
        super().__init__()
        self.conv1      = nn.Conv2d(in_channels, planes, kernel_size=1, bias=False)
        self.bn1        = nn.BatchNorm2d(planes)
        self.conv2      = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2        = nn.BatchNorm2d(planes)
        self.conv3      = nn.Conv2d(planes, planes * self.expansion, kernel_size=1, bias=False)
        self.bn3        = nn.BatchNorm2d(planes * self.expansion)
        self.relu       = nn.ReLU(inplace=True)
        self.dropout    = nn.Dropout(dropout)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.dropout(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out  = self.relu(out)
        return out


class CIFARResNet(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()

        # CIFAR-adapted stem: 3x3 conv, stride=1, no MaxPool
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        # Channel progression: 64->256 | 128->512 | 256->1024 | 512->2048
        self.layer1 = self._make_layer(64,   64,  2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(256,  128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(512,  256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(1024, 512, 2, stride=2, dropout=dropout)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Custom head -- identical to prior experiments; no Softmax (CrossEntropyLoss handles it)
        self.fc = nn.Sequential(
            nn.Linear(2048, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, 10)
        )

        self._init_weights()

    def _make_layer(self, in_channels, planes, blocks, stride=1, dropout=0.1):
        downsample = None
        if stride != 1 or in_channels != planes * Bottleneck.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, planes * Bottleneck.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * Bottleneck.expansion)
            )

        layers = [Bottleneck(in_channels, planes, stride, downsample, dropout)]
        in_channels = planes * Bottleneck.expansion
        for _ in range(1, blocks):
            layers.append(Bottleneck(in_channels, planes, dropout=dropout))

        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [5]:
model = CIFARResNet().to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters:     {total_p:,}")

# Forward pass verification
model.eval()
with torch.no_grad():
    dummy = torch.randn(32, 3, 32, 32).to(device)
    out   = model(dummy)
    print(f"\nForward pass output shape: {out.shape}")  # Expected: torch.Size([32, 10])

Trainable parameters: 14,209,674
Total parameters:     14,209,674

Forward pass output shape: torch.Size([32, 10])


## S4 - Shared Utilities

In [6]:
def evaluate_model(model, loader, criterion, device, label):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total
    accuracy = correct / total
    print(f"{label} | Test Loss: {avg_loss:.4f} | Test Acc: {accuracy:.4f} ({accuracy*100:.2f}%)")
    return avg_loss, accuracy, all_preds, all_labels


def per_class_accuracy(preds, labels_true, class_names):
    preds_np  = np.array(preds)
    labels_np = np.array(labels_true)

    print(f"{'Class':<12} {'Correct':>8} {'Total':>8} {'Acc':>8}")
    print("-" * 40)
    for i, cls in enumerate(class_names):
        mask    = labels_np == i
        correct = (preds_np[mask] == i).sum()
        total   = mask.sum()
        acc     = correct / total
        print(f"{cls:<12} {correct:>8} {total:>8} {acc:>8.2%}")

## S5 - Phase 1 Training -- 10k Dataset

### Hypothesis

In [ ]:
# Phase 1: 10k subset -- 80/20 split (8k train / 2k val)
torch.manual_seed(53)
indices_p1   = torch.randperm(len(full_train_dataset))[:10000].tolist()

split_p1     = int(0.8 * len(indices_p1))
train_idx_p1 = indices_p1[:split_p1]   # 8000
val_idx_p1   = indices_p1[split_p1:]   # 2000

train_subset_p1 = Subset(full_train_dataset,     train_idx_p1)
val_subset_p1   = Subset(full_train_dataset_val, val_idx_p1)

train_loader_p1 = DataLoader(
    train_subset_p1, batch_size=64, shuffle=True,
    num_workers=8, pin_memory=True, persistent_workers=True
)
val_loader_p1 = DataLoader(
    val_subset_p1, batch_size=64, shuffle=False,
    num_workers=8, pin_memory=True, persistent_workers=True
)
P1_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'scratch_phase1_best.pth')
print(f"Phase 1 -- Train: {len(train_subset_p1)} | Val: {len(val_subset_p1)}")

Phase 1 -- Train: 8000 | Val: 2000


In [8]:
# Fresh CIFARResNet -- all layers unfrozen (full simultaneous training)
model = CIFARResNet().to(device)

criterion    = nn.CrossEntropyLoss()
optimizer_p1 = optim.Adam(model.parameters(), lr=1e-3)

print(f"Optimizer: Adam lr=1e-3")
print(f"Criterion: CrossEntropyLoss")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Optimizer: Adam lr=1e-3
Criterion: CrossEntropyLoss
Trainable parameters: 14,209,674


In [ ]:
NUM_EPOCHS = 20
PATIENCE   = 3
MIN_DELTA  = 0.01

best_val_loss_p1 = float('inf')
patience_counter = 0
best_model_p1    = None

history_p1 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_p1:
        images, labels = images.to(device), labels.to(device)
        optimizer_p1.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer_p1.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # Validation phase
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader_p1:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            val_correct  += predicted.eq(labels).sum().item()
            val_total    += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct / val_total

    history_p1['train_loss'].append(train_loss)
    history_p1['train_acc'].append(train_acc)
    history_p1['val_loss'].append(val_loss)
    history_p1['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Patience: {patience_counter}/{PATIENCE}")

    if val_loss < best_val_loss_p1 - MIN_DELTA:
        best_val_loss_p1 = val_loss
        patience_counter = 0
        best_model_p1    = copy.deepcopy(model)
        print(f"  >> New best: {best_val_loss_p1:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

P1_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'scratch_phase1_best.pth')
P1_CHECKPOINT_TS = os.path.join(CHECKPOINT_DIR, f'scratch_phase1_best-{timestamp}.pth')

# Save consistent filename (used by Phase 2 loader)
torch.save({
    'weights': best_model_p1.state_dict(),
    'optimizer': optimizer_p1.state_dict(),
    'best_val_loss': best_val_loss_p1
}, P1_CHECKPOINT)

# Save timestamped backup
torch.save({
    'weights': best_model_p1.state_dict(),
    'optimizer': optimizer_p1.state_dict(),
    'best_val_loss': best_val_loss_p1
}, P1_CHECKPOINT_TS)

print(f"Phase 1 best weights saved: {P1_CHECKPOINT}")
print(f"Timestamped backup saved:   {P1_CHECKPOINT_TS}")

Epoch 01 | Train Loss: 4.3443 | Train Acc: 0.1440 | Val Loss: 2.2238 | Val Acc: 0.1790 | Patience: 0/3
  >> New best: 2.2238
Epoch 02 | Train Loss: 2.2211 | Train Acc: 0.1648 | Val Loss: 2.1802 | Val Acc: 0.1635 | Patience: 0/3
  >> New best: 2.1802
Epoch 03 | Train Loss: 2.1450 | Train Acc: 0.1695 | Val Loss: 2.1018 | Val Acc: 0.1675 | Patience: 0/3
  >> New best: 2.1018
Epoch 04 | Train Loss: 2.0955 | Train Acc: 0.1711 | Val Loss: 2.0201 | Val Acc: 0.1745 | Patience: 0/3
  >> New best: 2.0201
Epoch 05 | Train Loss: 2.0545 | Train Acc: 0.1750 | Val Loss: 1.9845 | Val Acc: 0.1745 | Patience: 0/3
  >> New best: 1.9845
Epoch 06 | Train Loss: 2.0244 | Train Acc: 0.1800 | Val Loss: 1.9644 | Val Acc: 0.1975 | Patience: 0/3
  >> New best: 1.9644
Epoch 07 | Train Loss: 1.9640 | Train Acc: 0.1883 | Val Loss: 1.9387 | Val Acc: 0.1900 | Patience: 0/3
  >> New best: 1.9387
Epoch 08 | Train Loss: 1.9506 | Train Acc: 0.1799 | Val Loss: 1.8863 | Val Acc: 0.1870 | Patience: 0/3
  >> New best: 1.8863


### Phase 1 continued cycles

In [12]:
# Phase 1 continued -- load best weights and continue training
P1_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'scratch_phase1_best.pth')
model = CIFARResNet().to(device)
model.load_state_dict(torch.load(P1_CHECKPOINT, weights_only=True))

criterion    = nn.CrossEntropyLoss()
optimizer_p1 = optim.Adam(model.parameters(), lr=1e-3)

print(f"Loaded: {P1_CHECKPOINT}")
print(f"Optimizer: Adam lr=1e-3")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

RuntimeError: Error(s) in loading state_dict for CIFARResNet:
	Missing key(s) in state_dict: "stem.0.weight", "stem.1.weight", "stem.1.bias", "stem.1.running_mean", "stem.1.running_var", "layer1.0.conv1.weight", "layer1.0.bn1.weight", "layer1.0.bn1.bias", "layer1.0.bn1.running_mean", "layer1.0.bn1.running_var", "layer1.0.conv2.weight", "layer1.0.bn2.weight", "layer1.0.bn2.bias", "layer1.0.bn2.running_mean", "layer1.0.bn2.running_var", "layer1.0.conv3.weight", "layer1.0.bn3.weight", "layer1.0.bn3.bias", "layer1.0.bn3.running_mean", "layer1.0.bn3.running_var", "layer1.0.downsample.0.weight", "layer1.0.downsample.1.weight", "layer1.0.downsample.1.bias", "layer1.0.downsample.1.running_mean", "layer1.0.downsample.1.running_var", "layer1.1.conv1.weight", "layer1.1.bn1.weight", "layer1.1.bn1.bias", "layer1.1.bn1.running_mean", "layer1.1.bn1.running_var", "layer1.1.conv2.weight", "layer1.1.bn2.weight", "layer1.1.bn2.bias", "layer1.1.bn2.running_mean", "layer1.1.bn2.running_var", "layer1.1.conv

#### Then Rerun training loop

In [16]:
# Best epoch verification -- reload checkpoint and confirm val_loss matches
checkpoint = torch.load(P1_CHECKPOINT, weights_only=True)
model.load_state_dict(checkpoint['weights'])
val_loss_verify, _, _, _ = evaluate_model(model, val_loader_p1, criterion, device, 'Phase 1 Verify')
print(f"Expected: {best_val_loss_p1:.4f} | Got: {val_loss_verify:.4f}")

Phase 1 Verify | Test Loss: 1.7775 | Test Acc: 0.2485 (24.85%)
Expected: 1.7775 | Got: 1.7775


In [31]:
_, test_acc_p1, preds_p1, labels_p1 = evaluate_model(
    model, test_loader, criterion, device, 'Phase 1 -- 10k'
)
per_class_accuracy(preds_p1, labels_p1, CLASS_NAMES)

Phase 1 -- 10k | Test Loss: 0.6161 | Test Acc: 0.8117 (81.17%)
Class         Correct    Total      Acc
----------------------------------------
airplane          890     1000   89.00%
automobile        953     1000   95.30%
bird              772     1000   77.20%
cat               636     1000   63.60%
deer              727     1000   72.70%
dog               684     1000   68.40%
frog              825     1000   82.50%
horse             880     1000   88.00%
ship              900     1000   90.00%
truck             850     1000   85.00%


### Phase 1 Observations

## S6 - Phase 2 Training -- 35k Dataset

### Hypothesis

In [13]:
# Phase 2: 35k subset -- 80/20 split (28k train / 7k val)
torch.manual_seed(53)
indices_p2   = torch.randperm(len(full_train_dataset))[:35000].tolist()

split_p2     = int(0.8 * len(indices_p2))
train_idx_p2 = indices_p2[:split_p2]   # 28000
val_idx_p2   = indices_p2[split_p2:]   # 7000

train_subset_p2 = Subset(full_train_dataset,     train_idx_p2)
val_subset_p2   = Subset(full_train_dataset_val, val_idx_p2)

train_loader_p2 = DataLoader(
    train_subset_p2, batch_size=128, shuffle=True,
    num_workers=6, pin_memory=True, persistent_workers=True
)
val_loader_p2 = DataLoader(
    val_subset_p2, batch_size=128, shuffle=False,
    num_workers=6, pin_memory=True, persistent_workers=True
)

print(f"Phase 2 -- Train: {len(train_subset_p2)} | Val: {len(val_subset_p2)}")

Phase 2 -- Train: 28000 | Val: 7000


In [14]:
# Load Phase 1 checkpoint as starting point -- all layers remain unfrozen
checkpoint = torch.load(P1_CHECKPOINT, weights_only=True)
model.load_state_dict(checkpoint['weights'])

optimizer_p2 = optim.AdamW(model.parameters(), lr=1e-4)

print(f"Loaded:    {P1_CHECKPOINT}")
print(f"Optimizer: AdamW lr=1e-4")

Loaded:    Q:\scripts\projects\ComputerVision-Term9\checkpoints\scratch_phase1_best.pth
Optimizer: AdamW lr=1e-4


In [ ]:
NUM_EPOCHS = 20
PATIENCE   = 3
MIN_DELTA  = 0.005

best_val_loss_p2 = float('inf')
patience_counter = 0
best_model_p2    = None

history_p2 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_p2:
        images, labels = images.to(device), labels.to(device)
        optimizer_p2.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer_p2.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader_p2:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            val_correct  += predicted.eq(labels).sum().item()
            val_total    += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct / val_total

    history_p2['train_loss'].append(train_loss)
    history_p2['train_acc'].append(train_acc)
    history_p2['val_loss'].append(val_loss)
    history_p2['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Patience: {patience_counter}/{PATIENCE}")

    if val_loss < best_val_loss_p2 - MIN_DELTA:
        best_val_loss_p2 = val_loss
        patience_counter = 0
        best_model_p2    = copy.deepcopy(model)
        print(f"  >> New best: {best_val_loss_p2:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

P2_CHECKPOINT    = os.path.join(CHECKPOINT_DIR, 'scratch_phase2_best.pth')
P2_CHECKPOINT_TS = os.path.join(CHECKPOINT_DIR, f'scratch_phase2_best-{timestamp}.pth')

torch.save({
    'weights': best_model_p2.state_dict(),
    'optimizer': optimizer_p2.state_dict(),
    'best_val_loss': best_val_loss_p2
}, P2_CHECKPOINT)

torch.save({
    'weights': best_model_p2.state_dict(),
    'optimizer': optimizer_p2.state_dict(),
    'best_val_loss': best_val_loss_p2
}, P2_CHECKPOINT_TS)

print(f"Phase 2 best weights saved: {P2_CHECKPOINT}")
print(f"Timestamped backup saved:   {P2_CHECKPOINT_TS}")

Epoch 01 | Train Loss: 1.7806 | Train Acc: 0.2575 | Val Loss: 1.7389 | Val Acc: 0.2790 | Patience: 0/3
  >> New best: 1.7389
Epoch 02 | Train Loss: 1.7585 | Train Acc: 0.2600 | Val Loss: 1.7609 | Val Acc: 0.2921 | Patience: 0/3
Epoch 03 | Train Loss: 1.7309 | Train Acc: 0.2687 | Val Loss: 1.6712 | Val Acc: 0.3036 | Patience: 1/3
  >> New best: 1.6712
Epoch 04 | Train Loss: 1.7228 | Train Acc: 0.2682 | Val Loss: 1.6882 | Val Acc: 0.2866 | Patience: 0/3
Epoch 05 | Train Loss: 1.6987 | Train Acc: 0.2837 | Val Loss: 1.6612 | Val Acc: 0.3067 | Patience: 1/3
  >> New best: 1.6612
Epoch 06 | Train Loss: 1.6649 | Train Acc: 0.3087 | Val Loss: 1.6079 | Val Acc: 0.3307 | Patience: 0/3
  >> New best: 1.6079
Epoch 07 | Train Loss: 1.6276 | Train Acc: 0.3169 | Val Loss: 1.5763 | Val Acc: 0.3436 | Patience: 0/3
  >> New best: 1.5763
Epoch 08 | Train Loss: 1.6085 | Train Acc: 0.3286 | Val Loss: 1.5713 | Val Acc: 0.3351 | Patience: 0/3
  >> New best: 1.5713
Epoch 09 | Train Loss: 1.5929 | Train Acc: 0

NameError: name 'P2_CHECKPOINT_TS' is not defined

In [19]:
checkpoint = torch.load(P2_CHECKPOINT, weights_only=True)
model.load_state_dict(checkpoint['weights'])
val_loss_verify, _, _, _ = evaluate_model(model, val_loader_p2, criterion, device, 'Phase 2 Verify')
print(f"Expected: {best_val_loss_p2:.4f} | Got: {val_loss_verify:.4f}")

Phase 2 Verify | Test Loss: 1.3673 | Test Acc: 0.5009 (50.09%)
Expected: 1.3673 | Got: 1.3673


In [32]:
_, test_acc_p2, preds_p2, labels_p2 = evaluate_model(
    model, test_loader, criterion, device, 'Phase 2 -- 35k'
)
per_class_accuracy(preds_p2, labels_p2, CLASS_NAMES)

Phase 2 -- 35k | Test Loss: 0.6161 | Test Acc: 0.8117 (81.17%)
Class         Correct    Total      Acc
----------------------------------------
airplane          890     1000   89.00%
automobile        953     1000   95.30%
bird              772     1000   77.20%
cat               636     1000   63.60%
deer              727     1000   72.70%
dog               684     1000   68.40%
frog              825     1000   82.50%
horse             880     1000   88.00%
ship              900     1000   90.00%
truck             850     1000   85.00%


### Phase 2 Observations

## S7 - Phase 3 Training -- 50k Dataset

### Hypothesis

In [20]:
# Phase 3: Full 50k training set -- 80/20 split (40k train / 10k val)
torch.manual_seed(53)
indices_p3   = torch.randperm(len(full_train_dataset)).tolist()

split_p3     = int(0.8 * len(indices_p3))
train_idx_p3 = indices_p3[:split_p3]   # 40000
val_idx_p3   = indices_p3[split_p3:]   # 10000

train_subset_p3 = Subset(full_train_dataset,     train_idx_p3)
val_subset_p3   = Subset(full_train_dataset_val, val_idx_p3)

train_loader_p3 = DataLoader(
    train_subset_p3, batch_size=128, shuffle=True,
    num_workers=6, pin_memory=True, persistent_workers=True
)
val_loader_p3 = DataLoader(
    val_subset_p3, batch_size=128, shuffle=False,
    num_workers=6, pin_memory=True, persistent_workers=True
)

print(f"Phase 3 -- Train: {len(train_subset_p3)} | Val: {len(val_subset_p3)}")

Phase 3 -- Train: 40000 | Val: 10000


In [21]:
# Load Phase 2 checkpoint as starting point -- all layers remain unfrozen
checkpoint = torch.load(P2_CHECKPOINT, weights_only=True)
model.load_state_dict(checkpoint['weights'])
optimizer_p3 = optim.AdamW(model.parameters(), lr=1e-5)
optimizer_p3.load_state_dict(checkpoint['optimizer'])

print(f"Loaded:    {P2_CHECKPOINT}")
print(f"Optimizer: AdamW lr=1e-5")

Loaded:    Q:\scripts\projects\ComputerVision-Term9\checkpoints\scratch_phase2_best.pth
Optimizer: AdamW lr=1e-5


#### Training Loop

In [22]:
NUM_EPOCHS = 20
PATIENCE   = 3
MIN_DELTA  = 0.005

best_val_loss_p3 = float('inf')
patience_counter = 0
best_model_p3    = None

history_p3 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_p3:
        images, labels = images.to(device), labels.to(device)
        optimizer_p3.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer_p3.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader_p3:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            val_correct  += predicted.eq(labels).sum().item()
            val_total    += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct / val_total

    history_p3['train_loss'].append(train_loss)
    history_p3['train_acc'].append(train_acc)
    history_p3['val_loss'].append(val_loss)
    history_p3['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Patience: {patience_counter}/{PATIENCE}")

    if val_loss < best_val_loss_p3 - MIN_DELTA:
        best_val_loss_p3 = val_loss
        patience_counter = 0
        best_model_p3    = copy.deepcopy(model)
        print(f"  >> New best: {best_val_loss_p3:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

P3_CHECKPOINT    = os.path.join(CHECKPOINT_DIR, 'scratch_phase3_best.pth')
P3_CHECKPOINT_TS = os.path.join(CHECKPOINT_DIR, f'scratch_phase3_best-{timestamp}.pth')

torch.save({
    'weights': best_model_p3.state_dict(),
    'optimizer': optimizer_p3.state_dict(),
    'best_val_loss': best_val_loss_p3
}, P3_CHECKPOINT)

torch.save({
    'weights': best_model_p3.state_dict(),
    'optimizer': optimizer_p3.state_dict(),
    'best_val_loss': best_val_loss_p3
}, P3_CHECKPOINT_TS)

print(f"Phase 3 best weights saved: {P3_CHECKPOINT}")
print(f"Timestamped backup saved:   {P3_CHECKPOINT_TS}")

Epoch 01 | Train Loss: 1.3356 | Train Acc: 0.4927 | Val Loss: 1.3300 | Val Acc: 0.4952 | Patience: 0/3
  >> New best: 1.3300
Epoch 02 | Train Loss: 1.3008 | Train Acc: 0.5074 | Val Loss: 1.2643 | Val Acc: 0.5228 | Patience: 0/3
  >> New best: 1.2643
Epoch 03 | Train Loss: 1.2722 | Train Acc: 0.5155 | Val Loss: 1.2564 | Val Acc: 0.5293 | Patience: 0/3
  >> New best: 1.2564
Epoch 04 | Train Loss: 1.2413 | Train Acc: 0.5263 | Val Loss: 1.2179 | Val Acc: 0.5407 | Patience: 0/3
  >> New best: 1.2179
Epoch 05 | Train Loss: 1.2184 | Train Acc: 0.5332 | Val Loss: 1.1745 | Val Acc: 0.5503 | Patience: 0/3
  >> New best: 1.1745
Epoch 06 | Train Loss: 1.1974 | Train Acc: 0.5423 | Val Loss: 1.1798 | Val Acc: 0.5443 | Patience: 0/3
Epoch 07 | Train Loss: 1.1671 | Train Acc: 0.5493 | Val Loss: 1.1471 | Val Acc: 0.5603 | Patience: 1/3
  >> New best: 1.1471
Epoch 08 | Train Loss: 1.1442 | Train Acc: 0.5584 | Val Loss: 1.1159 | Val Acc: 0.5683 | Patience: 0/3
  >> New best: 1.1159
Epoch 09 | Train Loss:

In [ ]:
# Verification of best model weights captured in checkpoint
checkpoint = torch.load(P3_CHECKPOINT, weights_only=True)
model.load_state_dict(checkpoint['weights'])
val_loss_verify, _, _, _ = evaluate_model(model, val_loader_p3, criterion, device, 'Phase 3 Verify')
print(f"Expected: {best_val_loss_p3:.4f} | Got: {val_loss_verify:.4f}")

Phase 3 Verify | Test Loss: 0.7594 | Test Acc: 0.7478 (74.78%)
Expected: 0.7594 | Got: 0.7594


In [ ]:
# Per class set evaluation
_, test_acc_p3, preds_p3, labels_p3 = evaluate_model(
    model, test_loader, criterion, device, 'Phase 3 -- 50k'
)
per_class_accuracy(preds_p3, labels_p3, CLASS_NAMES)

Phase 3 -- 50k | Test Loss: 0.7609 | Test Acc: 0.7543 (75.43%)
Class         Correct    Total      Acc
----------------------------------------
airplane          842     1000   84.20%
automobile        918     1000   91.80%
bird              677     1000   67.70%
cat               272     1000   27.20%
deer              586     1000   58.60%
dog               754     1000   75.40%
frog              873     1000   87.30%
horse             846     1000   84.60%
ship              903     1000   90.30%
truck             872     1000   87.20%


#### Phase 3 continuation  

- including gradient monitoring

In [26]:
# Load Phase 3 checkpoint as starting point -- all layers remain unfrozen
checkpoint = torch.load(P3_CHECKPOINT, weights_only=True)
model.load_state_dict(checkpoint['weights'])
optimizer_p3 = optim.AdamW(model.parameters(), lr=1e-5)
optimizer_p3.load_state_dict(checkpoint['optimizer'])

print(f"Loaded:    {P3_CHECKPOINT}")
print(f"Optimizer: AdamW lr=1e-5")
print(f"Best val loss to beat: {checkpoint['best_val_loss']:.4f}")

Loaded:    Q:\scripts\projects\ComputerVision-Term9\checkpoints\scratch_phase3_best.pth
Optimizer: AdamW lr=1e-5
Best val loss to beat: 0.7594


In [27]:
NUM_EPOCHS = 20
PATIENCE   = 3
MIN_DELTA  = 0.005

best_val_loss_p3 = checkpoint['best_val_loss']
patience_counter = 0
best_model_p3 = copy.deepcopy(model)

history_p3 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_p3:
        images, labels = images.to(device), labels.to(device)
        optimizer_p3.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        # Gradient monitoring -- record mean absolute gradient per layer group
        grad_monitor = {}
        for name, param in model.named_parameters():
            if param.grad is not None:
                for layer in ['stem', 'layer1', 'layer2', 'layer3', 'layer4', 'fc']:
                    if name.startswith(layer):
                        if layer not in grad_monitor:
                            grad_monitor[layer] = []
                        grad_monitor[layer].append(param.grad.abs().mean().item())
        optimizer_p3.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader_p3:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            val_correct  += predicted.eq(labels).sum().item()
            val_total    += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct / val_total

    history_p3['train_loss'].append(train_loss)
    history_p3['train_acc'].append(train_acc)
    history_p3['val_loss'].append(val_loss)
    history_p3['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Patience: {patience_counter}/{PATIENCE}")
    
    for layer, grads in grad_monitor.items():
        print(f"  {layer}: grad={np.mean(grads):.6f}")
    
    if val_loss < best_val_loss_p3 - MIN_DELTA:
        best_val_loss_p3 = val_loss
        patience_counter = 0
        best_model_p3    = copy.deepcopy(model)
        print(f"  >> New best: {best_val_loss_p3:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

P3_CHECKPOINT    = os.path.join(CHECKPOINT_DIR, 'scratch_phase3_best.pth')
P3_CHECKPOINT_TS = os.path.join(CHECKPOINT_DIR, f'scratch_phase3_best-{timestamp}.pth')

torch.save({
    'weights': best_model_p3.state_dict(),
    'optimizer': optimizer_p3.state_dict(),
    'best_val_loss': best_val_loss_p3
}, P3_CHECKPOINT)

torch.save({
    'weights': best_model_p3.state_dict(),
    'optimizer': optimizer_p3.state_dict(),
    'best_val_loss': best_val_loss_p3
}, P3_CHECKPOINT_TS)

print(f"Phase 3 best weights saved: {P3_CHECKPOINT}")
print(f"Timestamped backup saved:   {P3_CHECKPOINT_TS}")

Epoch 01 | Train Loss: 0.7284 | Train Acc: 0.7559 | Val Loss: 0.8297 | Val Acc: 0.7447 | Patience: 0/3
  stem: grad=0.061915
  layer1: grad=0.008183
  layer2: grad=0.003280
  layer3: grad=0.001671
  layer4: grad=0.001054
  fc: grad=0.006653
Epoch 02 | Train Loss: 0.7004 | Train Acc: 0.7661 | Val Loss: 0.7279 | Val Acc: 0.7702 | Patience: 1/3
  stem: grad=0.081079
  layer1: grad=0.007473
  layer2: grad=0.002846
  layer3: grad=0.001476
  layer4: grad=0.001039
  fc: grad=0.005218
  >> New best: 0.7279
Epoch 03 | Train Loss: 0.6704 | Train Acc: 0.7767 | Val Loss: 0.7686 | Val Acc: 0.7514 | Patience: 0/3
  stem: grad=0.056599
  layer1: grad=0.005907
  layer2: grad=0.002528
  layer3: grad=0.001431
  layer4: grad=0.001091
  fc: grad=0.005031
Epoch 04 | Train Loss: 0.6458 | Train Acc: 0.7843 | Val Loss: 0.7585 | Val Acc: 0.7665 | Patience: 1/3
  stem: grad=0.030962
  layer1: grad=0.004278
  layer2: grad=0.001840
  layer3: grad=0.001003
  layer4: grad=0.000679
  fc: grad=0.004971
Epoch 05 | Tra

In [28]:
# Verification of best model weights captured in checkpoint 
checkpoint = torch.load(P3_CHECKPOINT, weights_only=True)
model.load_state_dict(checkpoint['weights'])
val_loss_verify, _, _, _ = evaluate_model(model, val_loader_p3, criterion, device, 'Phase 3 Verify')
print(f"Expected: {best_val_loss_p3:.4f} | Got: {val_loss_verify:.4f}")

Phase 3 Verify | Test Loss: 0.6049 | Test Acc: 0.8121 (81.21%)
Expected: 0.6049 | Got: 0.6049


In [29]:
# Per class set evaluation
_, test_acc_p3, preds_p3, labels_p3 = evaluate_model(
    model, test_loader, criterion, device, 'Phase 3 -- 50k'
)
per_class_accuracy(preds_p3, labels_p3, CLASS_NAMES)

Phase 3 -- 50k | Test Loss: 0.6161 | Test Acc: 0.8117 (81.17%)
Class         Correct    Total      Acc
----------------------------------------
airplane          890     1000   89.00%
automobile        953     1000   95.30%
bird              772     1000   77.20%
cat               636     1000   63.60%
deer              727     1000   72.70%
dog               684     1000   68.40%
frog              825     1000   82.50%
horse             880     1000   88.00%
ship              900     1000   90.00%
truck             850     1000   85.00%


In [33]:
print("=" * 60)
print("CIFAR-10 From Scratch vs Transfer -- Results Comparison")
print("=" * 60)
print(f"{'Model':<35} {'Test Acc':>10}")
print("-" * 60)
print(f"{'S11-R1 Baseline (transfer, 35k)':<35} {'92.87%':>10}")
print(f"{'Phase 1 -- From Scratch, 10k':<35} {f'{test_acc_p1:.2%}':>10}")
print(f"{'Phase 2 -- From Scratch, 35k':<35} {f'{test_acc_p2:.2%}':>10}")
print(f"{'Phase 3 -- From Scratch, 50k':<35} {f'{test_acc_p3:.2%}':>10}")
print("=" * 60)

CIFAR-10 From Scratch vs Transfer -- Results Comparison
Model                                 Test Acc
------------------------------------------------------------
S11-R1 Baseline (transfer, 35k)         92.87%
Phase 1 -- From Scratch, 10k            81.17%
Phase 2 -- From Scratch, 35k            81.17%
Phase 3 -- From Scratch, 50k            81.17%


### Phase 3 Observations